# Isotope Effects: H₂O, HDO and D₂O

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4v2/blob/main/temp/06-isotope-effects.ipynb)
:::


In [1]:
####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style sheets and helper functions from the book's GitHub repository
if IN_COLAB:
    if not Path("styles/tufte.mplstyle").exists():
        !wget -q -N -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/tufte.mplstyle
        !wget -q -N https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/psi4_helpers.py
    plt.style.use("styles/tufte.mplstyle")
    nproc = 2          # Colab gives you 2 cores
else:
    sys.path.insert(0, "..")                    # psi4_helpers.py is one folder up
    plt.style.use("../styles/tufte.mplstyle")
    nproc = 4          # house limit: 4 threads

import psi4_helpers as ph

# All Psi4 .out and .log files go into the output folder
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Resources: house limits are 4 threads and 4 GB
psi4.set_num_threads(nproc)
_ = psi4.set_memory("2 GB")


  Threads set to 4 by Python driver.



  Memory set to   1.863 GiB by Python driver.


Replacing hydrogen with deuterium does not change the electrons at all. Within the **Born–Oppenheimer approximation**, H₂O, HDO and D₂O share exactly the same potential energy surface, the same equilibrium structure and the same electronic energy. Only the **masses** change, and with them the vibrational frequencies and zero-point energies. That is the origin of **kinetic isotope effects**, a key tool in physical organic chemistry.

## What You Will Learn
- How to specify **isotopes** in a *Psi4* geometry.
- Why one Hessian is enough for every isotopologue.
- How frequencies scale with mass, and why the H/D ratio is about 1.37 rather than $\sqrt{2}$.
- How a difference in **zero-point energy** gives a simple estimate of a primary kinetic isotope effect.

Experimental fundamentals (cm⁻¹):

| Molecule | bend | O–D stretch | O–H stretch |
|---|---|---|---|
| H₂O | 1595 | — | 3657 (sym), 3756 (antisym) |
| HDO | 1402 | 2727 | 3707 |
| D₂O | 1178 | 2671 (sym), 2788 (antisym) | — |

In [2]:
psi4.set_output_file("output/06-isotope-effects.out", False);
psi4.set_options({"basis": "6-31g*", "scf_type": "df"})
h2kJ = psi4.constants.hartree2kJmol

## 1. One Hessian for Normal Water

```{note} Psi4 settings
:class: dropdown
`b3lyp/6-31g*`
: The same affordable level as the *Thermochemistry* notebook. Isotope **ratios** and **differences** are less sensitive to the level of theory than absolute frequencies, because the errors largely cancel. *Alternatives:* `mp2/cc-pvtz` or `ccsd(t)/cc-pvtz` for frequencies close to the harmonic experimental values (see the *Vibrations* notebook).
```

In [3]:
water = psi4.geometry("""
0 1
O
H 1 0.96
H 1 0.96 2 104.5
""")
psi4.optimize("b3lyp", molecule=water)
E, wfn = psi4.frequency("b3lyp", molecule=water, return_wfn=True)

Optimizer: Optimization complete!


## 2. Building the Isotopologues

*Psi4* reads an isotope as `symbol@mass`. Deuterium is `H@2.014101778`. We write the **optimized** coordinates of normal water with new labels, keeping the structure exactly the same.

```{note} Psi4 settings in the geometry
:class: dropdown
`H@2.014101778`
: A hydrogen atom with the mass of deuterium (in u). Any mass can be given, for example `H@3.01604928` for tritium or `O@17.9991610` for ¹⁸O. The *Psi4* manual warns that isotope support is limited: isotopes must be set when the molecule is created, and they are used mainly in frequency analysis.

`units bohr`, `no_com`, `no_reorient`
: We copy the coordinates straight from `water.geometry()`, which are in bohr and already in *Psi4*'s standard orientation, so we tell *Psi4* not to convert or move them. This keeps them in the same frame as the Hessian.

Symmetry
: HDO has lower symmetry ($C_s$) than H₂O and D₂O ($C_{2v}$). *Psi4* detects this automatically from the masses.
```

In [4]:
xyz = water.geometry().np      # bohr

def isotopologue(labels):
    """Return a Psi4 molecule with the optimized water structure and new atom labels."""
    lines = ["0 1", "units bohr", "no_com", "no_reorient"]
    for label, (x, y, z) in zip(labels, xyz):
        lines.append(f"{label} {x:.10f} {y:.10f} {z:.10f}")
    mol = psi4.geometry("\n".join(lines))
    mol.update_geometry()
    return mol

D = "H@2.014101778"
species = {
    "H2O": ["O", "H", "H"],
    "HDO": ["O", D, "H"],
    "D2O": ["O", D, D],
}

## 3. Frequencies and Zero-Point Energies

`psi4.driver.vibanal_wfn(wfn, molecule=...)` reuses the Hessian stored in `wfn` with the masses of the new molecule. No new quantum chemistry is done.

In [5]:
rows = []
for name, labels in species.items():
    mol = isotopologue(labels)
    vib = psi4.driver.vibanal_wfn(wfn, molecule=mol)
    is_vib = np.array([t == "V" for t in vib["TRV"].data])
    f = np.sort(vib["omega"].data[is_vib].real)
    rows.append({"molecule": name, "point group": mol.schoenflies_symbol(),
                 "bend": f[0], "stretch 1": f[1], "stretch 2": f[2],
                 "ZPVE (kJ/mol)": psi4.variable("ZPVE") * h2kJ})

iso_table = pd.DataFrame(rows)
iso_table.round(1)

,molecule,point group,bend,stretch 1,stretch 2,ZPVE (kJ/mol)
0,H2O,c2v,1713.2,3727.7,3849.4,55.6
1,HDO,cs,1501.6,2751.0,3791.3,48.1
2,D2O,c2v,1253.6,2687.8,2819.5,40.4


Every frequency drops when H is replaced by D, and the ZPVE falls by about 7–8 kJ/mol per deuterium. In HDO the two stretches are no longer mixed: one is almost a pure O–H stretch and the other an almost pure O–D stretch.

## 4. Why the Ratio is Not $\sqrt{2}$

For a simple spring, $\nu = \frac{1}{2\pi}\sqrt{k/\mu}$ where $\mu$ is the **reduced mass**. The force constant $k$ is the same for O–H and O–D. If the oxygen were infinitely heavy, $\mu$ would equal the mass of H or D and the ratio would be $\sqrt{2} = 1.414$. Oxygen is not infinitely heavy, so:

$$
\frac{\nu_{\mathrm{OH}}}{\nu_{\mathrm{OD}}} = \sqrt{\frac{\mu_{\mathrm{OD}}}{\mu_{\mathrm{OH}}}}, \qquad \mu = \frac{m_{\mathrm{O}}\, m_{\mathrm{H}}}{m_{\mathrm{O}} + m_{\mathrm{H}}}
$$

In [6]:
mO, mH, mD = 15.99491462, 1.00782503, 2.01410178
mu_OH = mO * mH / (mO + mH)
mu_OD = mO * mD / (mO + mD)
print(f"Reduced-mass prediction:  nu(OH)/nu(OD) = {np.sqrt(mu_OD / mu_OH):.3f}")

hdo = iso_table.set_index("molecule").loc["HDO"]
print(f"Calculated for HDO:       nu(OH)/nu(OD) = {hdo['stretch 2'] / hdo['stretch 1']:.3f}")
print(f"Experiment for HDO:       nu(OH)/nu(OD) = {3707 / 2727:.3f}")

Reduced-mass prediction:  nu(OH)/nu(OD) = 1.374
Calculated for HDO:       nu(OH)/nu(OD) = 1.378
Experiment for HDO:       nu(OH)/nu(OD) = 1.359


## 5. A Simple Kinetic Isotope Effect

Suppose a reaction breaks an O–H (or O–D) bond in the rate-determining step, and the stretching vibration disappears in the transition state. The O–H bond starts with more zero-point energy, so it needs less extra energy to reach the transition state. The difference in activation energy is the difference in stretching ZPE, $\tfrac{1}{2}h c\,(\tilde\nu_{\mathrm{OH}} - \tilde\nu_{\mathrm{OD}})$, and the maximum primary isotope effect is

$$
\frac{k_{\mathrm{H}}}{k_{\mathrm{D}}} \approx \exp\!\left(\frac{\tfrac{1}{2}h c\,(\tilde\nu_{\mathrm{OH}} - \tilde\nu_{\mathrm{OD}})}{k_{\mathrm{B}} T}\right)
$$

This simple model ignores bending modes, tunnelling and the vibrations of the transition state, so treat it as an upper estimate.

In [7]:
c2 = 1.438777          # second radiation constant hc/k_B in cm K
T = 298.15
d_nu = hdo["stretch 2"] - hdo["stretch 1"]          # O-H minus O-D stretch, cm-1
kie = np.exp(0.5 * d_nu * c2 / T)
print(f"Difference in stretching ZPE: {0.5 * d_nu:.0f} cm-1  = {0.5 * d_nu * 0.01196266:.1f} kJ/mol")
print(f"Estimated maximum kH/kD at {T} K: {kie:.1f}")

Difference in stretching ZPE: 520 cm-1  = 6.2 kJ/mol
Estimated maximum kH/kD at 298.15 K: 12.3


Observed primary isotope effects for O–H and C–H bond breaking are usually between about 2 and 8 at room temperature, below this simple maximum because some zero-point energy remains in the transition state.

```{tip} Going further
The same approach works for any molecule: calculate one Hessian, then call `vibanal_wfn()` with isotopically labelled copies. For a real reaction you would also calculate the **transition state** and its frequencies for both isotopologues.
```

## Summary
- Isotopes are given as `symbol@mass` in the geometry.
- Isotopologues share one potential energy surface, so one Hessian serves them all.
- Frequencies scale with the reduced mass; the O–H/O–D ratio is about 1.37.
- The difference in zero-point energy explains primary kinetic isotope effects.